In [ ]:

import pandas as pd
# ====================== 你的配置（完全不用修改） ======================
STOCK_FILE = "/mnt/c/Users/M259941/Desktop/Desktop/320.Office Document/326.Python/material inventory/Export_2026-04-16_04-36-45.xlsx"
SEARCH_FILE = "/mnt/c/Users/M259941/Desktop/Desktop/320.Office Document/326.Python/material inventory/search list.xlsx"
OUTPUT_FILE = "/mnt/c/Users/M259941/Desktop/Desktop/320.Office Document/326.Python/material inventory/weight output.xlsx"
NAME_COL = "Material Names"    # 库存表B列
WEIGHT_COL = "Current amount"  # 库存表AE列
CONTAINER_COL = "Container ID" # 库存表R列
# ======================================================================
# 读取待查材料（仅读取A列搜索词）
search_df = pd.read_excel(SEARCH_FILE, engine="openpyxl")
material_list = search_df.iloc[:, 0].dropna().astype(str).tolist()
print(f"✅ 读取待查材料：{len(material_list)} 个")
# 读取库存表
stock_df = pd.read_excel(STOCK_FILE, engine="openpyxl")
print("✅ 读取库存表成功")
# 固定表头（最终版）
columns = [
    "待查材料名称", "匹配状态", "库存材料名称(Material Names)",
    "匹配结果1", "匹配结果2", "匹配结果3", "匹配结果4", "匹配结果5",
    "匹配结果6", "匹配结果7", "匹配结果8", "匹配结果9", "匹配结果10",
    "匹配结果11", "匹配结果12", "匹配结果13", "匹配结果14", "匹配结果15",
    "总重量"  # S列 固定总重量
]
result_data = []
# 核心匹配逻辑
for material in material_list:
    # 初始化固定19列：A待查名 B状态 C库存名 D-R结果 S总重量
    row = [material, "未找到", ""] + [""] * 15 + [0.0]
    
    # 模糊匹配
    mask = stock_df[NAME_COL].str.contains(material, na=False)
    matched = stock_df[mask].copy()
    
    if matched.empty:
        result_data.append(row)
        print(f"❌ {material}：未找到")
        continue
    # 过滤0重量
    matched[WEIGHT_COL] = pd.to_numeric(matched[WEIGHT_COL], errors="coerce").fillna(0)
    valid = matched[matched[WEIGHT_COL] > 0]
    if valid.empty:
        row[1] = "重量为0，已跳过"
        result_data.append(row)
        print(f"⏭️ {material}：重量为0")
        continue
    # 🔥 核心：C列写入【库存表匹配到的 Material Names】
    row[1] = "已找到"
    row[2] = valid[NAME_COL].iloc[0]  # 取库存表的材料名称
    
    # 填充匹配结果 + 计算总重量
    total_weight = 0.0
    for i, (_, r) in enumerate(valid.iterrows()):
        if i >= 15: break
        ctn = str(r[CONTAINER_COL]) if pd.notna(r[CONTAINER_COL]) else "无ID"
        w = float(r[WEIGHT_COL])
        row[3 + i] = f"{ctn}: {w}"
        total_weight += w
    row[-1] = total_weight
    result_data.append(row)
    print(f"✅ {material} | 库存名称：{row[2]} | 总重量：{total_weight}")
# 导出Excel
df = pd.DataFrame(result_data, columns=columns)
df.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")
print("\n" + "="*80)
print("🎉 导出成功！")
print("📊 最终格式：")
print("A列=待查材料 | B列=状态 | C列=库存表Material Names | S列=总重量")
print(f"📁 文件位置：{OUTPUT_FILE}")
print("="*80)
